**This notebook processes raw data collected from behavioral trials to create a clean `tadpoles_behavior.csv` file for analysis in `larvae_metamorphosis.R`.**

Run with

* Python 3.8
* pandas 2.0.3
* numpy 1.24.3

In [1]:
# Import standard libraries
import os
import re

# Import published packages from PyPI
import pandas as pd
import numpy as np

**Specify directories**

In [2]:
DATA_IN = 'datasets/raw/tadpoles_behavior'
DATA_OUT = 'datasets/processed'

# Load raw data

In [3]:
video_info_df = pd.read_csv(
    os.path.join(DATA_IN, 'video_info.csv'), 
    index_col = False
)
startstop_times_df = pd.read_csv(
    os.path.join(DATA_IN, 'startstop_times.csv'), 
    index_col = False
)

# Data wrangling

In [4]:
video_attributes_set = {
    'day', 
    'memory_card', 
    'camera_datetime', 
    'video_name',
    'blinded_sequence_number', 
    'pond', 
    'clutch', 
    'arena_room', 
    'temperature', 
    'photoperiod',
    'antipredator_cue', 
    'location_type', 
    'all_minutes_video_duration',
    'num_individuals'
}

**Store in a new column the converted video durations in minutes:seconds to minutes**

In [5]:
video_info_df['video_duration'].unique()

array(['29:58:00', '13:46', '28:01:00', '13:06', '25:27:00', '29:18:00',
       '26:17:00', '26:24:00'], dtype=object)

In [6]:
video_info_df[[
    'video_duration_minutes', 
    'video_duration_seconds'
]] = video_info_df['video_duration'].str.extract(
    r'(?P<video_duration_minutes>\d+):(?P<video_duration_seconds>\d+)',
    expand = True
)
video_info_df = video_info_df.astype({
    'video_duration_minutes': 'float32',
    'video_duration_seconds': 'float32'
}, copy = False)
video_info_df[
    'all_minutes_video_duration'
] = video_info_df['video_duration_minutes'] + video_info_df[
    'video_duration_seconds'
] / 60

**Merge video attributes in `video_info_df` with raw data in `startstop_times_df`**

In [7]:
startstop_times_design_df = startstop_times_df.merge(
    video_info_df, 
    how = "left", 
    validate = "many_to_one"
)

**Add columns about trial setup encoded in video filenames**

In [8]:
startstop_times_design_df[[
    'blinded_sequence_number', 
    'pond', 
    'clutch', 
    'arena_room', 
    'antipredator_stimulus'
]] = startstop_times_design_df['video_name'].str.extract(
    r'T(?P<blinded_sequence_number>\d)_p(?P<pond>.*)_C(?P<clutch>\d)_(?P<arena_room>\d{3}\D)_(?P<antipredator_stimulus>[^_]+)_*.*.MOV',
    expand = True
)
startstop_times_design_df = startstop_times_design_df.astype({
    'blinded_sequence_number': 'int8'
}, copy = False)

In [9]:
# revises & adds more column names about trial design
startstop_times_design_df['location_type'] = startstop_times_design_df[
    'arena_room'
].map(
    {'210F': 'in_rearing', '227A': 'in_rearing', '210G': 'out_novel'}
)
startstop_times_design_df['antipredator_cue'] = startstop_times_design_df[
    'antipredator_stimulus'
].map(
    {'cue': 'homogenate', 'control': 'water'}
)
startstop_times_design_df['temperature'] = np.where(
    startstop_times_design_df['blinded_sequence_number'] <= 2, 
    'early', 
    'late'
)
startstop_times_design_df['photoperiod'] = np.where(
    startstop_times_design_df['blinded_sequence_number'] % 2 != 0, 
    'early', 
    'late'
)

**Store in new columns the converted start and stop times in minutes:seconds to minutes**

In [10]:
startstop_times_design_df[[
    'time_start_feeding_minutes', 
    'time_start_feeding_seconds'
]] = startstop_times_design_df['time_start_feeding'].str.extract(
    r'(?P<time_start_feeding_minutes>\d+):(?P<time_start_feeding_seconds>\d+)',
    expand = True
)

startstop_times_design_df[[
    'time_stop_feeding_minutes', 
    'time_stop_feeding_seconds'
]] = startstop_times_design_df['time_stop_feeding'].str.extract(
    r'(?P<time_stop_feeding_minutes>\d+):(?P<time_stop_feeding_seconds>\d+)',
    expand = True
)

startstop_times_design_df[[
    'time_start_under_lettuce_minutes', 
    'time_start_under_lettuce_seconds'
]] = startstop_times_design_df['time_start_under_lettuce'].str.extract(
    r'(?P<time_start_under_lettuce_minutes>\d+):(?P<time_start_under_lettuce_seconds>\d+)',
    expand = True
)

startstop_times_design_df[[
    'time_end_under_lettuce_minutes', 
    'time_end_under_lettuce_seconds'
]] = startstop_times_design_df['time_end_under_lettuce'].str.extract(
    r'(?P<time_end_under_lettuce_minutes>\d+):(?P<time_end_under_lettuce_seconds>\d+)',
    expand = True
)

startstop_times_design_df = startstop_times_design_df.astype({
    'time_start_feeding_minutes': 'float32',
    'time_start_feeding_seconds': 'float32', 
    'time_stop_feeding_minutes': 'float32', 
    'time_stop_feeding_seconds': 'float32', 
    'time_start_under_lettuce_minutes': 'float32', 
    'time_start_under_lettuce_seconds': 'float32', 
    'time_end_under_lettuce_minutes': 'float32', 
    'time_end_under_lettuce_seconds': 'float32'
}, copy = False)

startstop_times_design_df[
    'minutes_until_start_feeding'
] = startstop_times_design_df[
    'time_start_feeding_minutes'
] + startstop_times_design_df['time_start_feeding_seconds'] / 60

startstop_times_design_df[
    'minutes_until_stop_feeding'
] = startstop_times_design_df[
    'time_stop_feeding_minutes'
] + startstop_times_design_df['time_stop_feeding_seconds'] / 60

startstop_times_design_df[
    'minutes_until_start_under_lettuce'
] = startstop_times_design_df[
    'time_start_under_lettuce_minutes'
] + startstop_times_design_df['time_start_under_lettuce_seconds'] / 60

startstop_times_design_df[
    'minutes_until_end_under_lettuce'
] = startstop_times_design_df[
    'time_end_under_lettuce_minutes'
] + startstop_times_design_df['time_end_under_lettuce_seconds'] / 60

## Compute durations under lettuce

**For each observed instance across trials**

In [11]:
startstop_times_design_df['minutes_duration_under_lettuce'] = startstop_times_design_df[
    'minutes_until_end_under_lettuce'
] - startstop_times_design_df['minutes_until_start_under_lettuce']

In [12]:
# number of instances <1 second
startstop_times_design_df[startstop_times_design_df[
    'minutes_duration_under_lettuce'
] == 0].shape[0]

26

In [13]:
# if hiding occurred <1 second, treat as lasting half a second 
#  (units in fractions of a minute)
startstop_times_design_df[
    'minutes_duration_under_lettuce'
] = startstop_times_design_df['minutes_duration_under_lettuce'].map(
    lambda x: 1 / 120 if x == 0 else x
)

**Summed per trial**

In [14]:
durations_under_lettuce_s = startstop_times_design_df.groupby([
    'blinded_sequence_number', 
    'temperature', 
    'photoperiod', 
    'location_type', 
    'antipredator_cue', 
    'pond', 
    'clutch', 
    'video_name'
]
)['minutes_duration_under_lettuce'].sum()
durations_under_lettuce_df = durations_under_lettuce_s.to_frame().reset_index()
durations_under_lettuce_df.rename(columns = {
    'minutes_duration_under_lettuce': 'summed_minutes_under_lettuce_per_trial'
}, inplace = True)

# Combine summed durations under lettuce with presence/absence of feeding & start times of feeding

In [15]:
# select trials with feeding present
startstop_times_design_feeding_present_df = startstop_times_design_df[
    startstop_times_design_df['time_start_feeding'].notna()
]
startstop_times_design_feeding_present_df = startstop_times_design_feeding_present_df.loc[
    startstop_times_design_feeding_present_df.groupby(
        'video_name'
    ).minutes_until_start_feeding.idxmin()
].reset_index(drop = True)
video_names_feeding_present = set(startstop_times_design_feeding_present_df[
    'video_name'
])

In [16]:
# select trials without feeding
startstop_times_design_feeding_absent_df = startstop_times_design_df.loc[
    ~startstop_times_design_df['video_name'].isin(video_names_feeding_present)
].drop_duplicates(subset = 'video_name')

In [17]:
assert(
    startstop_times_design_feeding_present_df.shape[0] + \
    startstop_times_design_feeding_absent_df.shape[0] == \
    video_info_df.shape[0]
)

In [18]:
# Mark presence/absence of feeding
startstop_times_design_feeding_present_df['feed'] = 1
startstop_times_design_feeding_absent_df['feed'] = 0

In [19]:
# merge with summed durations under lettuce
trials_df = pd.concat([
    startstop_times_design_feeding_present_df, 
    startstop_times_design_feeding_absent_df
], ignore_index = True)
trials_df = trials_df.filter(items = list(video_attributes_set) + [
    'minutes_until_first_feed',
    'feed'
])
trials_df = trials_df.merge(
    durations_under_lettuce_df.filter(items = [
        'video_name',
        'summed_minutes_under_lettuce_per_trial'
    ]), 
    how = 'inner', 
    validate = '1:1'
)

In [20]:
trials_df.shape

(93, 16)

In [21]:
# write to csv file for analysis in R
trials_df.to_csv(os.path.join(DATA_OUT, "tadpoles_behavior.csv"), index = False)